# Executive Exploratory Data Analysis

## Georgia Tech MSA Spring 2026 Practicum

This is your EDA Executive Summary. Fill in the details of your analysis here.

# Executive Summary: How Influential is a player on the pitch?
**Project Phase:** Exploratory Data Analysis (EDA)  
**Objective:** To define, measure, and visualize player influence across global soccer leagues using interactive data analytics; for each game, across the the season, at the position level.

---

## 1. The Vision
The goal of this project is to move beyond surface-level statistics to identify the **most influential player** on the pitch. While traditional metrics often favor goal-scorers, this project seeks to quantify influence by analyzing how a player’s presence and their specific actions shifts the momentum and outcome of a game.

### Defining "Influence"
To find the true "needle-movers," and beyond the EDA, I seek to understand:
* **Volume & Involvement:** How often do they touch the ball in high-leverage areas?
* **Presence vs. Absence:** What happens to team performance when the player is off the pitch?
* **Positioning & Momentum:** How does their spatial positioning correlate with shifts in match control?
* **Efficiency:** Consistency in contributing to key metrics like Expected Goals ($xG$) and pass accuracy.

---

## 2. EDA: Understanding the Foundation
Before i start looking at the individuals, I must first understand the team dynamics that lead to success. The initial analysis focuses on the relationship between team-level **Independent Variables** (like passing and $xG$) and the **Dependent Variable** (Match Outcome: Win, Loss, or Draw).

### Key Visualization 1: The Match Performance Profile
* **Focus:** Granular, match-by-match event distribution.
* **Data Points:** For every unique `Match ID`, I visualize the distribution of event types, total $xG$, pass count, and pass accuracy.
* **My Goal:** To see the "shape" of a win versus the "shape" of a draw or loss. Does high pass accuracy actually correlate with points, or is $xG$ the only reliable predictor based on the data I have?

### Key Visualization 2: The Seasonal Comparative Index
* **Focus:** Consistency and dominance across the entire season.
* **The "Green/Red" Indicator:** A comparative view that pits a team’s stats against their opponent for every match.
    * <span style="color:green">**Green:**</span> Team outperformed the opponent in that specific metric (e.g., higher pass count).
    * <span style="color:red">**Red:**</span> Opponent outperformed the team.
* **My Goal:** To identify patterns of dominance. Does a team consistently "out-pass" the opposition but still lose? This helps isolate where individual player influence might be the missing link.

---

## 3. Assumptions & Methodology
* **Beyond the Scoreline:** I recognize that a draw can still contain a "most influential" performance. I still plan to weigh contributions that lead to high-quality chances, even if the final score doesn't reflect it.
* **Rating Neutrality:** I am intentionally ignoring traditional match ratings to avoid inherent biases, relying instead on raw event data and positioning.
* **Scalability:** A user should be able to filter by country, leages, etc..

## 4. Interactive EDA Dashboard
I went ahead and apply my logic above. As shown below, anyone can already at a high level analyze each team, for each game, and see if they can spot a trend.

![Alt text](interactive_dashboard.jpg)

When I take a look at a team, like Barcelone for example, I can see that they have a high percentage of wins, it quite often come with higher expected goals. 


# Parquet Output Data Schema

This document specifies the schema for the denormalized Parquet files stored in `parq_output/`. These files are optimized for analytical queries by reducing the need for complex joins.

## 1. `matches.parquet`
**Description**: Consolidated match metadata joined with competition details and team names.
**Grain**: One row per match.

| Column | Type | Description |
| :--- | :--- | :--- |
| `match_id` | INTEGER | Unique match identifier |
| `match_date` | VARCHAR | Date of the match (YYYY-MM-DD) |
| `home_team` | VARCHAR | Name of the home team |
| `away_team` | VARCHAR | Name of the away team |
| `home_score` | INTEGER | Final score for the home team |
| `away_score` | INTEGER | Final score for the away team |
| `competition_name`| VARCHAR | Name of the competition (e.g., "La Liga") |
| `season_name` | VARCHAR | Season name (e.g., "2023/2024") |
| `gender` | VARCHAR | Competition gender |
| `is_youth` | BOOLEAN | Flag for youth competitions |
| `is_international`| BOOLEAN | Flag for international competitions |
| `stadium` | VARCHAR | Name of the stadium |
| `referee` | VARCHAR | Name of the referee |
| `kickoff` | VARCHAR | Match kickoff time |

---

## 2. `events.parquet`
**Description**: Detailed match events (passes, shots, etc.). This table is inherently denormalized with player and team names.
**Grain**: One row per event.

| Column | Type | Description |
| :--- | :--- | :--- |
| `id` | VARCHAR | Unique event UUID |
| `match_id` | INTEGER | Reference to `matches.parquet` |
| `period` | INTEGER | Match period (1, 2, 3, 4) |
| `minute` | INTEGER | Minute of the event |
| `second` | INTEGER | Second of the event |
| `type` | VARCHAR | Event type (Pass, Shot, Carry, etc.) |
| `team` | VARCHAR | Team performing the action |
| `player` | VARCHAR | Player performing the action |
| `location_x` | FLOAT | X coordinate (0-120) |
| `location_y` | FLOAT | Y coordinate (0-80) |
| `play_pattern` | VARCHAR | Context of play (e.g., "From Corner") |
| `shot_statsbomb_xg`| FLOAT | Expected goals value (for Shots) |
| `pass_outcome` | VARCHAR | Outcome of the pass (Incomplete, etc.) |

---

## 3. `lineups.parquet`
**Description**: Consolidated player match participation, joining rosters with position history and disciplinary actions.
**Grain**: One row per player-position-match-card combination.

| Column | Type | Description |
| :--- | :--- | :--- |
| `match_id` | INTEGER | Reference to `matches.parquet` |
| `team_name` | VARCHAR | Team name |
| `player_name` | VARCHAR | Player's full name |
| `jersey_number` | INTEGER | Player's jersey number for the match |
| `position_name` | VARCHAR | Name of the position played |
| `from_time` | VARCHAR | Time player started in this position |
| `to_time` | VARCHAR | Time player ended in this position |
| `card_type` | VARCHAR | Type of card received (Yellow, Red) |
| `card_reason` | VARCHAR | Reason for the disciplinary action |

---

## 4. `three_sixty.parquet`
**Description**: High-resolution spatial tracking data for players present at the moment of an event.
**Grain**: One row per tracked player per frame.

| Column | Type | Description |
| :--- | :--- | :--- |
| `event_uuid` | VARCHAR | Reference to `events.parquet` |
| `match_id` | INTEGER | Reference to `matches.parquet` |
| `teammate` | BOOLEAN | Whether the tracked player is a teammate of the actor |
| `actor` | BOOLEAN | Whether the tracked player is the person doing the event |
| `keeper` | BOOLEAN | Whether the tracked player is the goalkeeper |
| `location_x` | FLOAT | X coordinate of the player |
| `location_y` | FLOAT | Y coordinate of the player |
| `visible_area` | VARCHAR | JSON polygon representing the camera's field of view |

---

## 5. `reference.parquet`
**Description**: Consolidated lookup table for all categorical entities (Teams, Players, Positions, etc.).
**Grain**: One row per entity.

| Column | Type | Description |
| :--- | :--- | :--- |
| `table_name` | VARCHAR | Source lookup table (e.g., 'team', 'player', 'position') |
| `id` | INTEGER | Original unique ID from the lookup table |
| `name` | VARCHAR | The human-readable name of the entity |
| `extra_info` | VARCHAR | Additional metadata (e.g., team gender) |

---

## 📜 License & Attribution

- **Data Source**: StatsBomb Open Data
- **License**: [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)
- **Usage**: Usage of StatsBomb Open Data is subject to their license (non-commercial use only, attribution required).
- **Citation**: "StatsBomb Open Data"